# 03 — 4ft-Miner: Úloha 1
## Výkonnost žen na extrémních vzdálenostech
### 4IZ503 Projektový seminář — Ultra Marathon Running

---

### Slovní zadání

Mají ženy na extrémních vzdálenostech (100+ mil) nadprůměrnou pravděpodobnost
být v rychlé třetině startovního pole?

**Hypotéza:** Gender gap se zmenšuje s délkou závodu. Ženy jsou fyziologicky
lépe vybaveny pro ultra-long distance — efektivnější tukový metabolismus,
lepší termoregulace a psychická odolnost při extrémní zátěži.

**Ante:** `gender(F) ∧ distance_cat(vzdálenost)`
**Succ:** `speed_cat(rychlý)`

**Porovnání:** Totéž pravidlo pro muže — kde je rozdíl největší?

**Business interpretace:** Pokud ženy relativně dominují na extrémních
vzdálenostech, organizátoři by měli cílit marketing na ženskou komunitu
právě u 100mi+ závodů.

---

### Parametry úlohy

| Parametr | Hodnota |
|---|---|
| Procedura | 4ft-Miner |
| Base (min. počet záznamů) | 500 |
| AAD (min. odchylka od průměru) | 0.02 |
| Ante | gender(subset) ∧ distance_cat(subset), maxlen=2 |
| Succ | speed_cat(rychlý) |
| Data | ultra_clean_cm.parquet (~6.87M záznamů) |

> ⚠️ **Metodická poznámka:** `speed_cat` je počítán per event —
> každý závod má ~33 % rychlých závodníků. AAD (Above Average Deviation)
> měří odchylku confidence pravidla od průměrného podílu rychlých v celém datasetu.
> AAD > 0 znamená nadprůměrný podíl rychlých, AAD < 0 podprůměrný.


## 1. Import a načtení dat

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from cleverminer import cleverminer
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('../data/processed')

df_cm = pd.read_parquet(DATA_DIR / 'ultra_clean_cm.parquet')
print(f"Načteno: {len(df_cm):,} řádků")
print(f"Sloupce: {df_cm.columns.tolist()}")


Cleverminer version  1.2.6
Načteno: 6,877,109 řádků
Sloupce: ['speed_cat', 'age_group', 'experience_cat', 'gender', 'distance_cat', 'season', 'surface', 'elevation_cat']


## 2. Příprava dat pro úlohu

In [2]:
# Pro tuto úlohu potřebujeme: gender, distance_cat, speed_cat
cols = ['gender', 'distance_cat', 'speed_cat']
df_task = df_cm[cols].dropna().copy()

print(f"Záznamy s kompletními daty: {len(df_task):,}")
print()
print("Rozložení gender:")
print(df_task['gender'].value_counts())
print()
print("Rozložení distance_cat:")
print(df_task['distance_cat'].value_counts())
print()
print("Rozložení speed_cat (ověření ~33/33/33):")
print((df_task['speed_cat'].value_counts() / len(df_task) * 100).round(1))


Záznamy s kompletními daty: 6,516,919

Rozložení gender:
gender
M    5209410
F    1307509
Name: count, dtype: int64

Rozložení distance_cat:
distance_cat
kratka      2723310
stredni     1954909
dlouha      1321871
casovy       444433
extremni      72396
Name: count, dtype: int64

Rozložení speed_cat (ověření ~33/33/33):
speed_cat
střední    33.8
pomalý     33.3
rychlý     32.9
Name: count, dtype: float64


## 3. CleverMiner — 4ft-Miner úloha

In [3]:
cm = cleverminer(df=df_task)

cm.mine(
    proc='4ftMiner',
    quantifiers={'Base': 500, 'aad': 0.02},
    ante={
        'attributes': [
            {'name': 'gender',       'type': 'subset', 'minlen': 1, 'maxlen': 1},
            {'name': 'distance_cat', 'type': 'subset', 'minlen': 1, 'maxlen': 1},
        ],
        'minlen': 2, 'maxlen': 2, 'type': 'con'
    },
    succ={
        'attributes': [
            {'name': 'speed_cat', 'type': 'one', 'value': 'rychlý'}
        ],
        'minlen': 1, 'maxlen': 1, 'type': 'con'
    }
)

print("\nSouhrn:")
cm.print_summary()


Cleverminer version 1.2.6.
Starting data preparation ...
Automatically reordering numeric categories ...
Automatically reordering numeric categories ...done
Encoding columns into bit-form...
Encoding columns into bit-form...done
Data preparation finished.
Will go for  4ftMiner
Starting to mine rules.
  0%|                                                    |Elapsed Time: 0:00:00
 15%|########                                            |Elapsed Time: 0:00:00
 39%|####################                                |Elapsed Time: 0:00:00
100%|####################################################|Elapsed Time: 0:00:00
Done. Total verifications : 10, rules 5, times: prep 6.07sec, processing 0.16sec

Souhrn:

CleverMiner task processing summary:

Task type : 4ftMiner
Number of verifications : 10
Number of rules : 5
Total time needed : 00h 00m 06s
Time of data preparation : 00h 00m 06s
Time of rule mining : 00h 00m 00s



## 4. Výsledky

In [4]:
print("Všechna pravidla (seřazená dle AAD):")
cm.print_rulelist(sortby='aad', storesorted=True)


Všechna pravidla (seřazená dle AAD):

List of rules:
RULEID BASE  CONF  AAD    Rule
     4 774697 0.371 +0.128 gender(M) & distance_cat(kratka) => speed_cat(rychlý) | ---
     5 571359 0.354 +0.075 gender(M) & distance_cat(stredni) => speed_cat(rychlý) | ---
     2 386205 0.346 +0.052 gender(M) & distance_cat(dlouha) => speed_cat(rychlý) | ---
     1 111479 0.341 +0.035 gender(M) & distance_cat(casovy) => speed_cat(rychlý) | ---
     3 21232 0.337 +0.025 gender(M) & distance_cat(extremni) => speed_cat(rychlý) | ---



In [6]:
n = cm.get_rulecount()
print(f"Počet pravidel: {n}")

for i in range(1, n+1):
    print(f"\n--- Pravidlo {i} ---")
    cm.print_rule(i)
    print(f"Quantifiers: {cm.get_quantifiers(i)}")
    try:
        print(f"Variables: {cm.get_rule_variables(i, 'ante', 'cat')}")
    except Exception as e:
        print(f"get_rule_variables error: {e}")
    try:
        print(f"Categories: {cm.get_rule_categories(i, 'ante', 'cat', 0)}")
    except Exception as e:
        print(f"get_rule_categories error: {e}")

Počet pravidel: 5

--- Pravidlo 1 ---


Rule id : 4

Base : 774697  Relative base : 0.119  CONF : 0.371  AAD : +0.128  BAD : -0.128

Cedents:
  antecedent : gender(M) & distance_cat(kratka)
  succcedent : speed_cat(rychlý)
  condition  : ---

Fourfold table
    |  S  |  ¬S |
----|-----|-----|
 A  |774697|1313687|
----|-----|-----|
¬A  |1369233|3059302|
----|-----|-----|

Quantifiers: {'base': 774697, 'rel_base': 0.11887473206280452, 'conf': 0.37095524577855415, 'aad': 0.12759525234682534, 'bad': -0.12759525234682534, 'fourfold': [774697, 1313687, 1369233, 3059302]}
Variables: ['gender', 'distance_cat']
ERROR: variable not found: ante,cat. Possible variables are ['gender', 'distance_cat', 'speed_cat']
get_rule_categories error: name 'exit' is not defined

--- Pravidlo 2 ---


Rule id : 5

Base : 571359  Relative base : 0.088  CONF : 0.354  AAD : +0.075  BAD : -0.075

Cedents:
  antecedent : gender(M) & distance_cat(stredni)
  succcedent : speed_cat(rychlý)
  condition  : ---

Fourfold t

## 5. Extrakce pravidel pro analýzu

In [ ]:
import re

rules = []
n = cm.get_rulecount()

for i in range(1, n + 1):
    quant = cm.get_quantifiers(i)
    rule_text = cm.get_ruletext(i)
    
    # Parsování z textu pravidla např: "gender(M) & distance_cat(kratka) => speed_cat(rychlý)"
    gender_match   = re.search(r'gender\((\w+)\)', rule_text)
    distance_match = re.search(r'distance_cat\((\w+)\)', rule_text)
    
    rules.append({
        'rule_id':  i,
        'gender':   gender_match.group(1)   if gender_match   else None,
        'distance': distance_match.group(1) if distance_match else None,
        'base':     quant.get('base'),
        'conf':     quant.get('conf'),
        'aad':      quant.get('aad'),
    })

df_rules = pd.DataFrame(rules)
print(f"Extrahováno {len(df_rules)} pravidel")
print()
print(df_rules.sort_values('aad', ascending=False).to_string(index=False))

TypeError: cleverminer.get_rule_variables() got multiple values for argument 'get_names'

## 6. Vizualizace

In [ ]:
# Seřadit distance kategoricky
dist_order = ['kratka', 'stredni', 'dlouha', 'extremni', 'casovy']
dist_labels = {'kratka': '<60 km', 'stredni': '60-100 km',
               'dlouha': '100-170 km', 'extremni': '>170 km', 'casovy': 'časový'}

df_rules['distance_ord'] = pd.Categorical(
    df_rules['distance'], categories=dist_order, ordered=True
)
df_rules = df_rules.sort_values('distance_ord')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('4ft-Miner: Výkonnost žen vs. mužů dle délky závodu\n'
             '(succ: speed_cat = rychlý, per event)',
             fontsize=13, fontweight='bold')

# Graf 1 — Confidence (podíl rychlých) dle gender × distance
df_f = df_rules[df_rules['gender'] == 'F'].set_index('distance_ord')['conf']
df_m = df_rules[df_rules['gender'] == 'M'].set_index('distance_ord')['conf']

dist_cats = [d for d in dist_order if d in df_rules['distance'].values]
x = np.arange(len(dist_cats))
width = 0.35

bars_f = ax1.bar(x - width/2,
                 [df_f.get(d, 0) for d in dist_cats],
                 width, label='Ženy (F)', color='tomato', alpha=0.85, edgecolor='white')
bars_m = ax1.bar(x + width/2,
                 [df_m.get(d, 0) for d in dist_cats],
                 width, label='Muži (M)', color='steelblue', alpha=0.85, edgecolor='white')

# Referenční linie — průměrný podíl rychlých (~33%)
ax1.axhline(0.333, color='black', linewidth=1, linestyle='--', label='Průměr (33%)')

for bars in [bars_f, bars_m]:
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax1.text(bar.get_x() + bar.get_width()/2, h + 0.003,
                     f'{h:.3f}', ha='center', va='bottom', fontsize=8)

ax1.set_xlabel('Kategorie vzdálenosti', fontsize=11)
ax1.set_ylabel('Confidence (podíl rychlých)', fontsize=11)
ax1.set_title('Podíl rychlých závodníků dle pohlaví a vzdálenosti', fontsize=11)
ax1.set_xticks(x)
ax1.set_xticklabels([dist_labels.get(d, d) for d in dist_cats], fontsize=9)
ax1.legend(fontsize=10)
ax1.grid(axis='y', alpha=0.3)
ax1.set_ylim(0.25, 0.45)

# Graf 2 — AAD: odchylka od průměru
df_aad_f = df_rules[df_rules['gender'] == 'F'].set_index('distance_ord')['aad']
df_aad_m = df_rules[df_rules['gender'] == 'M'].set_index('distance_ord')['aad']

bars_f2 = ax2.bar(x - width/2,
                  [df_aad_f.get(d, 0) for d in dist_cats],
                  width, label='Ženy (F)', color='tomato', alpha=0.85, edgecolor='white')
bars_m2 = ax2.bar(x + width/2,
                  [df_aad_m.get(d, 0) for d in dist_cats],
                  width, label='Muži (M)', color='steelblue', alpha=0.85, edgecolor='white')

ax2.axhline(0, color='black', linewidth=0.8, linestyle='--')

for bars in [bars_f2, bars_m2]:
    for bar in bars:
        h = bar.get_height()
        if abs(h) > 0.001:
            ax2.text(bar.get_x() + bar.get_width()/2,
                     h + 0.001 if h >= 0 else h - 0.003,
                     f'{h:+.3f}', ha='center',
                     va='bottom' if h >= 0 else 'top', fontsize=8)

ax2.set_xlabel('Kategorie vzdálenosti', fontsize=11)
ax2.set_ylabel('AAD (odchylka od průměru)', fontsize=11)
ax2.set_title('AAD: nadprůměrnost rychlých závodníků\n(kladné = nadprůměr, záporné = podprůměr)',
              fontsize=11)
ax2.set_xticks(x)
ax2.set_xticklabels([dist_labels.get(d, d) for d in dist_cats], fontsize=9)
ax2.legend(fontsize=10)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(DATA_DIR / '03_zeny_vzdalenosti.png', dpi=150, bbox_inches='tight')
plt.show()
print("Graf uložen.")


## 7. Interpretace grafů

### 📊 Interpretace grafů

**Graf vlevo — Confidence (podíl rychlých):**

Ukazuje jaký podíl závodníků dané skupiny (gender × vzdálenost) skončil
v rychlé třetině startovního pole. Přerušovaná linie = průměr 33 %.

Klíčové pozorování:
- Muži dosahují nadprůměrné confidence na krátkých a středních vzdálenostech
- Ženy mají nižší confidence celkově — ale rozdíl se s délkou závodu mění

**Graf vpravo — AAD (Above Average Deviation):**

AAD měří odchylku od průměrného podílu rychlých v celém datasetu.
Čím méně záporné (nebo kladné) AAD u žen, tím menší gender gap.

Klíčový nález: Pokud AAD žen na extrémních vzdálenostech je méně záporné
než na krátkých, hypotéza je potvrzena — gender gap se zmenšuje s délkou závodu.


## 8. Zajímavá pravidla

In [ ]:
print("=== ZAJÍMAVÁ PRAVIDLA ===")
print()

# Pravidla seřazená dle AAD
df_sorted = df_rules.sort_values('aad', ascending=False)

print("TOP 3 pravidla (nejvyšší AAD — nadprůměrný podíl rychlých):")
for _, row in df_sorted.head(3).iterrows():
    cm.print_rule(int(row['rule_id']))
    print()

print("BOTTOM 3 pravidla (nejnižší AAD — podprůměrný podíl rychlých):")
for _, row in df_sorted.tail(3).iterrows():
    cm.print_rule(int(row['rule_id']))
    print()


## 9. Souhrn a business interpretace

In [ ]:
print("=" * 60)
print("SOUHRN — Úloha 1: Výkonnost žen na extrémních vzdálenostech")
print("=" * 60)
print()

if len(df_rules) > 0:
    # Porovnání AAD pro ženy na různých vzdálenostech
    f_rules = df_rules[df_rules['gender'] == 'F'].copy()
    if len(f_rules) > 0:
        f_rules = f_rules.sort_values('distance_ord')
        print("AAD pro ženy dle vzdálenosti:")
        for _, row in f_rules.iterrows():
            label = dist_labels.get(row['distance'], row['distance'])
            print(f"  {label:15s}: AAD = {row['aad']:+.3f}  (conf = {row['conf']:.3f}, n = {row['base']:,.0f})")
        print()

        # Porovnání ženy vs muži na extremni
        f_ext = df_rules[(df_rules['gender']=='F') & (df_rules['distance']=='extremni')]
        m_ext = df_rules[(df_rules['gender']=='M') & (df_rules['distance']=='extremni')]

        if len(f_ext) > 0 and len(m_ext) > 0:
            diff = f_ext.iloc[0]['aad'] - m_ext.iloc[0]['aad']
            print(f"Rozdíl AAD (F vs M) na extremni vzdálenosti: {diff:+.3f}")
            if diff > 0:
                print("→ Ženy jsou relativně SILNĚJŠÍ na extrémních vzdálenostech ✓")
            else:
                print("→ Hypotéza se nepotvrdila — muži jsou relativně silnější i zde")

print()
print("BUSINESS DOPORUČENÍ:")
print("  Pokud je hypotéza potvrzena:")
print("  → Marketing 100mi+ závodů cílit na ženskou komunitu")
print("  → Komunikovat: 'Čím delší závod, tím více záleží na")
print("    vytrvalosti a psychice — kde ženy dominují'")
print("  → Ženské vlny / ženské kategorie na ultra závodech")


## Shrnutí

**Metoda:** 4ft-Miner (CleverMiner 1.2.6). Pravidla tvaru
`gender(X) ∧ distance_cat(Y)` ⟹ `speed_cat(rychlý)`.
Kvantifikátory: Base ≥ 500, AAD ≥ 0.02.

**Data:** ~6.87M závodníků, speed_cat per event.

**Klíčový nález:** [doplnit po spuštění]

**Limitace:**
- speed_cat per event znamená že každý závod má ~33% rychlých —
  srovnáváme relativní výkonnost v rámci závodu, ne absolutní rychlost
- Dataset má 19.8% žen — menší statistická síla pro ženské podskupiny
- athlete_id není globálně unikátní

**Další notebook:** `04_4ft_uloha2.ipynb` — Veteráni na časových závodech
